In [ ]:
import sympy as sp
import numpy as np
from scipy.special import jv
from scipy.sparse import lil_matrix
import matplotlib.pyplot as plt
from scipy.linalg import block_diag
from scipy.optimize import root
import sys
import os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
import Plot_setting
from Plot_setting import *

Hamiltonian $$H = \omega_a(x_a^2/2+p_a^2/2) + \omega_b(x_b^2/2+p_b^2/2) + \varepsilon_d\cos(\omega_d t)\sqrt 2 x_b-2E_J\sin(\varepsilon_p)[\sin(\sqrt 2 \varphi_ax_a+ \sqrt 2 \varphi_bx_b)-\sqrt 2 \varphi_ax_a- \sqrt 2 \varphi_bx_b]$$

EOMs
\begin{equation}
\begin{cases}
\dot x_a = \omega_a p_a \\
\dot p_a = -\omega_a x_a +2E_J\sin(\varepsilon_p)\sqrt 2 \varphi_a[\cos(\sqrt 2 \varphi_ax_a+ \sqrt 2 \varphi_bx_b)-1] \\
\dot x_b = \omega_b p_b \\
\dot p_b = -\omega_b x_b - \sqrt 2 \varepsilon_d\cos(\omega_d t)+2E_J\sin(\varepsilon_p)\sqrt 2 \varphi_b[\cos(\sqrt 2 \varphi_ax_a+ \sqrt 2 \varphi_bx_b)-1] -\kappa_b p_b\\
\end{cases}
\end{equation}

We rescale time and the variables
\begin{equation}
\begin{cases}
 dx_a^{\lambda}/d(2\tau) = p_a^{\lambda} \\
dp_a^{\lambda}/d(2\tau) = - x_a^{\lambda} +2E_J\sin(\varepsilon_p)\sqrt 2 \varphi_a^2 [\cos(\sqrt 2 x_a^{\lambda}+ \sqrt 2 x_b^{\lambda})-1]/\omega_a \\
dx_b^{\lambda}/d(4\tau) = p_b^{\lambda} \\
dp_b^{\lambda}/d(4\tau) = -x_b^{\lambda} - \sqrt 2 \varepsilon_d\varphi_b\cos(4\tau)/\omega_b+2E_J\sin(\varepsilon_p)\sqrt 2 \varphi_b^2 [\cos(\sqrt 2 x_a^{\lambda}+ \sqrt 2 x_b^{\lambda})-1]/\omega_b-\kappa_b p_b^{\lambda}/\omega_b\\
\end{cases}
\end{equation}

\begin{equation}
\begin{cases}
 dx_a^{\lambda}/d(2\tau) = p_a^{\lambda} \\
dp_a^{\lambda}/d(2\tau) = - x_a^{\lambda} +\sqrt 2\beta_a' [\cos(\sqrt 2 x_a^{\lambda}+ \sqrt 2 x_b^{\lambda})-1] \\
dx_b^{\lambda}/d(4\tau) = p_b^{\lambda} \\
dp_b^{\lambda}/d(4\tau) = -x_b^{\lambda} - \sqrt 2 \varepsilon_d'\cos(4\tau)+\sqrt 2\beta_b'[\cos(\sqrt 2 x_a^{\lambda}+ \sqrt 2 x_b^{\lambda})-1]-\kappa_b' p_b^{\lambda}\\
\end{cases}
\end{equation}

Finally we write the second order equations on the positions

\begin{equation}
\begin{cases}
\frac{1}{4}\frac{d^2x_a^{\lambda}}{d\tau^2} = - x_a^{\lambda} +\sqrt 2\beta_a' [\cos(\sqrt 2 x_a^{\lambda}+ \sqrt 2 x_b^{\lambda})-1] \\
\frac{1}{16}\frac{d^2x_b^{\lambda}}{d\tau^2} = -x_b^{\lambda} - \sqrt 2 \varepsilon_d'\cos(4\tau)+\sqrt 2\beta_b'[\cos(\sqrt 2 x_a^{\lambda}+ \sqrt 2 x_b^{\lambda})-1] -\frac{\kappa_b'}{2} dx_b^{\lambda}/d\tau\\
\end{cases}
\end{equation}

We introduce $\bold X = (x_a^{\lambda}, x_b^{\lambda})^T$ and we get

\begin{equation}
\begin{split}
\begin{pmatrix}1/4&0\\0&1/16\end{pmatrix}\ddot{\bold X}+\begin{pmatrix}0&0\\0&\kappa_b'/2\end{pmatrix}\dot{\bold X}+\begin{pmatrix}1&0\\0&1\end{pmatrix}\bold X-\begin{pmatrix}\beta_a'\\\beta_b'\end{pmatrix}\sqrt 2[\cos(\sqrt 2 x_a^{\lambda}+ \sqrt 2 x_b^{\lambda})-1] =& -\begin{pmatrix}0\\1\end{pmatrix}\sqrt 2 \varepsilon_d'\cos(4\tau)\\
M\ddot{\bold X} + C\dot{\bold X} + K\bold X + \bold f_{nl}(\bold X)= \bold f_{ext}(\omega = 4,\tau)
\end{split}
\end{equation}

Then we are going to write the position as fFourier series
$$\bold X(t)=\frac{\bold c_0^x}{\sqrt 2}+\sum_{k=1}^{N_H}\biggl(\bold s_k^x\sin(kt)+\bold c_k^x\cos(kt)\biggr)$$

Let $$\bold f_{nl}(\bold X) = \frac{\bold c_0^f}{\sqrt 2}+\sum_{k=1}^{N_H}\biggl(\bold s_k^f\sin(kt)+\bold c_k^f\cos(kt)\biggr)$$

Let $z = [\bold c_0^x, \bold s_1^x, \bold c_1^x, ...]$ and $b = [\bold c_0^f, \bold s_1^f, \bold c_1^f, ...]$

In [ ]:
h_bar=1
e=1
phi_quantum=1/2

#### Caracterization of the circuit #####

w_a = 25.338776456203686
w_b = 2*w_a

phi_a = 0.11
Z_a = 2*phi_quantum**2*phi_a**2
C_a_inv = Z_a*w_a
E_Ca = C_a_inv/2

phi_b = 0.204
Z_b = 2*phi_quantum**2*phi_b**2
C_b_inv = Z_b*w_b
E_Cb = C_b_inv/2

E_J = 37.12*2*np.pi*1.5

#Environment
kappa_b=1/10.4; # 1/ns from Leon's paper

w_d=w_b # Drive frequency
w_p=2*w_a-w_d # Pump Frequency

epsilon_p = .3
g2 = jv(1,epsilon_p)*E_J*phi_a**2*phi_b # (Leon's paper)
epsilon_d = 10.12*g2#0.75*g2_max

beta_a_p, beta_b_p = 2*E_J*np.sin(epsilon_p)*phi_a**2/(w_a), 2*E_J*np.sin(epsilon_p)*phi_b**2/(w_b)
B = np.array([beta_a_p, beta_b_p])  # shape (2,)
eps_d_p = epsilon_d*phi_b/w_b
kappa_b_p = kappa_b / w_b


# Given numerical values
b_0_norm = 0.33636495810132455
a_0_norm = 0.47287700802978716
b_m1_norm = -0.07201944643601312

# Parameters (must already be defined somewhere in your code)
# beta_b_ATS, beta_a, eps_d, lambda_b, omega_b_ATS

theta_b_0 = -0.141756683610786
theta_a_0_1 = theta_b_0 / 2
theta_b_m1 = 3 * theta_a_0_1

# Complex amplitudes
b_0 = b_0_norm * np.exp(1j * theta_b_0)
a_0 = a_0_norm * np.exp(1j * theta_a_0_1)
b_m1 = b_m1_norm * np.exp(1j * theta_b_m1)
a_1 = -0.06895043097048979
b_2 = beta_b_p * a_1 / beta_a_p

z_0_init = [-2*np.real(a_1), -2*np.real(b_2),
        0.01, 0.01, 0.01, 0.01,
        np.sqrt(2)*np.real(a_0), 0.0, -np.sqrt(2)*np.imag(a_0),0.0, 
        0.01, 0.01, 0.01, 0.01,
        0.0, np.sqrt(2)*np.imag(b_0), 0.0, np.sqrt(2)*np.real(b_0), 
        0.01, 0.01, 0.01, 0.01,
        0.0, np.sqrt(2)*np.real(b_m1), 0.0, -np.sqrt(2)*np.imag(b_m1),
        0.01, 0.01, 0.01, 0.01,
        0.0, 0.0, 0.0, 0.0]
# Cat-state values
#b_0_cat_state = 0.0

#a_0_cat_state_1 = 1j * np.sqrt(eps_d_p/ beta_b_p)
#z_0_init = [0.0, 0.0,
#        np.sqrt(2)*np.imag(a_0_cat_state_1), -0.1, 
#        np.sqrt(2)*np.real(a_0_cat_state_1), -0.1]
re_a = -0.4736842105263158
im_a = 0.894736842105263
#z_0_init = [0.1, 0.1, im_a, 0.0, re_a, 0.0, 0.0, im_a, 0.0, re_a, 0.0, 0.1, 0.0, 0.1]
z_0_init = np.array([ 1.09656245e-01,  1.88572491e-01,  6.36702210e-03,  8.76868903e-03,
         6.72303249e-03,  9.24024143e-03,  2.40775936e-01, -5.19021061e-17,
         1.29455705e-02,  2.55713260e-16, -1.36877767e-03,  6.68868671e-03,
         1.13576613e-03, -5.62391811e-03,  2.66417683e-04, -2.31832679e-02,
        -4.52413407e-03,  3.62199837e-01, -1.04515526e-04, -1.66109841e-03,
        -1.20309391e-04, -1.94500316e-03, -5.97721955e-03, -6.57545126e-02,
        -5.72870310e-04, -6.60436919e-03, -3.64503331e-06, -3.42114325e-05,
         7.05922864e-07,  6.51141406e-06,  2.84776137e-04,  2.49632530e-03,
        -2.19458382e-03, -1.88634782e-02])

z_0_init = np.array([ 1.90883953e-01,  3.28257297e-01,
                     0,0,0,0,
                     8.15731793e-01, -4.01663153e-16, 4.63736148e-02,  1.47628826e-16,
                     0,0,0,0,  
                     2.43166519e-04, -7.51702636e-02, -2.30869022e-03,  6.61178886e-01,
                     0,0,0,0,
                     -1.03785075e-02, -1.14179502e-01, -1.77819584e-03, -1.98305860e-02, 
                     0,0,0,0, 
                     7.90256579e-04,  6.83208442e-03, -3.41987001e-03, -2.93965874e-02])
z_0_init = np.array([ 2.55185083e-01,  4.38833985e-01,
       0,0,0,0,
        1.55009096e+00, -5.39630647e-16,2.60472650e-01, -5.14078703e-17, 
        0,0,0,0, 
        4.28238464e-04, -4.02341994e-01,-1.28235270e-03,  1.16439645e+00, 
        0,0,0,0,
       -1.10675108e-02, -1.21655990e-01,-6.03074114e-03, -6.66506303e-02, 
       0,0,0,0, 
       1.84798830e-03,  1.59151973e-02,-2.35175049e-03, -2.02010357e-02])


In [ ]:
def build_L(M, C, K, N_H, omega=1.0, nu=1.0):
    """
    Build linear HB operator L for ordering:
    [c0, s1, c1, s2, c2, ..., s_NH, c_NH] with each block in R^n.
    Returns L of shape (n*(2N_H+1), n*(2N_H+1)).
    """
    n = M.shape[0]
    dim = n * (2*N_H + 1)
    L = np.zeros((dim, dim), dtype=float)

    # helper to locate block indices in z
    def idx_c0():      return 0
    def idx_sk(k):     return 1 + 2*(k-1)
    def idx_ck(k):     return 2 + 2*(k-1)

    # DC: (K) acting on c0 (with your 1/sqrt2 convention, L is still K here
    # because the same scaling appears in Gamma and Gamma^+; keep L=K)
    i0 = idx_c0()
    L[i0*n:(i0+1)*n, i0*n:(i0+1)*n] = K

    # Harmonics
    for k in range(1, N_H+1):
        A = K - ((k*omega/nu)**2)*M  # n×n
        B = (k*omega/nu) * C          # n×n

        is_ = idx_sk(k)
        ic_ = idx_ck(k)

        # sine row blocks
        L[is_*n:(is_+1)*n, is_*n:(is_+1)*n] = A
        L[is_*n:(is_+1)*n, ic_*n:(ic_+1)*n] = -B

        # cosine row blocks
        L[ic_*n:(ic_+1)*n, is_*n:(is_+1)*n] = B
        L[ic_*n:(ic_+1)*n, ic_*n:(ic_+1)*n] = A

    return L

def build_b_ext(N_H, eps_d_p, nu = 2):
    """
    b_ext in the same ordering as z:
    [c0, s1, c1, s2, c2, ..., s_NH, c_NH], each block in R^2.
    Forcing: f_ext = -sqrt(2)*eps_d' * cos(2 tau) * e2.
    """
    n = 2
    b = np.zeros(n*(2*N_H+1), dtype=float)

    def idx_ck(k): return 2 + 2*(k-1)

    if N_H >= nu:
        ic2 = idx_ck(nu)
        # e2 = [0,1]
        b[ic2*n:(ic2+1)*n] = np.array([0.0, -np.sqrt(2.0)*eps_d_p])

    return b

In [ ]:
def build_Gamma(t, N_H, n, omega=2.0, nu=2.0):
    """
    Build Gamma = I_n ⊗ Phi with omega/nu = 1.

    t   : array of shape (N,)
    N_H : number of harmonics
    n   : number of DOFs
    """
    t = np.asarray(t)
    N = t.size
    k = np.arange(1, N_H + 1)

    # N x N_H
    S = np.sin(np.outer(t, k*omega/nu))
    C = np.cos(np.outer(t, k*omega/nu))

    # Build Phi: N x (2*N_H + 1) with columns [1/sqrt2, sin(1t), cos(1t), sin(2t), cos(2t), ...]
    Phi = np.empty((N, 2 * N_H + 1), dtype=float)
    Phi[:, 0] = 1.0 / np.sqrt(2.0)
    Phi[:, 1::2] = S
    Phi[:, 2::2] = C

    # Gamma: (n*N) x (n*(2*N_H+1))
    Gamma = np.kron(Phi, np.eye(n))
    return Gamma

def x_tilde_to_X(tildex, n, N):
    # tildex is time-major: [x(t1); x(t2); ...], each x(tj) is length n
    return tildex.reshape(N, n).T   # -> (n, N)

def X_to_x_tilde(X):
    # X is (n, N); return time-major vector
    return X.T.reshape(-1)

def f_nl_time(X):
    """
    X: shape (n, N) with n=2
    returns Fnl: shape (n, N)
    """
    y = np.sqrt(2.0) * (X[0, :] + X[1, :])          # shape (N,)
    g = np.cos(y) - 1.0                             # shape (N,)
    Fnl = -np.sqrt(2.0) * B[:, None] * g[None, :]    # (2,N)
    return Fnl

def b_nl(z, Gamma, Gamma_pinv, n, N):
    x_tilde = Gamma @ z
    X = x_tilde_to_X(x_tilde, n, N)
    Fnl = f_nl_time(X)              # (n,N)
    f_tilde = X_to_x_tilde(Fnl)     # (nN,)
    return Gamma_pinv @ f_tilde     # (n*(2N_H+1),)

In [ ]:
def residual(z, L, b_ext, Gamma, Gamma_pinv, n, N):
    return L @ z - b_nl(z, Gamma, Gamma_pinv, n, N) - b_ext

def dfnl_dX_blocks(X):
    """
    returns list of N blocks J_j = ∂f_nl/∂X at each time sample
    each block is 2x2
    """
    y = np.sqrt(2.0) * (X[0, :] + X[1, :])
    s = np.sin(y)  # shape (N,)

    # J_j = 2 sin(y_j) * [[beta_a, beta_a],[beta_b, beta_b]]
    base = np.array([[beta_a_p, beta_a_p],
                     [beta_b_p, beta_b_p]])  # 2x2

    J_blocks = [(2.0 * s[j]) * base for j in range(X.shape[1])]
    return J_blocks

def build_dftilde_dx_tilde(X, n, N):
    """
    Block diagonal (nN x nN) with n×n blocks per time sample,
    consistent with Gamma = kron(Phi, I_n) (time-major stacking):
      tilde x = [x(t1); x(t2); ...], each x(tj) in R^n.
    """
    J_blocks = dfnl_dX_blocks(X)  # length N, each (n,n)
    Jbig = lil_matrix((n*N, n*N), dtype=float)

    for j in range(N):
            Jj = J_blocks[j]
            rows = slice(j*n, (j+1)*n)
            cols = slice(j*n, (j+1)*n)
            Jbig[rows, cols] = Jj

    return Jbig.tocsr()

def db_dz(z, Gamma, Gamma_pinv, n, N):
    # time samples
    x_tilde = Gamma @ z
    X = x_tilde_to_X(x_tilde, n, N)

    # build ∂f/∂x in time
    Jbig = build_dftilde_dx_tilde(X, n, N)

    # chain rule
    return (Gamma_pinv @ (Jbig @ Gamma))

def jacobian(z, L, Gamma, Gamma_pinv, n, N):
    return L - db_dz(z, Gamma, Gamma_pinv, n, N)
def newton_solve(z0, L, b_ext, Gamma, Gamma_pinv, n, N, tol=1e-10, maxit=50):
    z = z0.copy()
    for it in range(maxit):
        r = residual(z, L, b_ext, Gamma, Gamma_pinv, n, N)
        norm_r = np.linalg.norm(r)
        if norm_r < tol:
            print("Converged", it, norm_r)
            return z

        J = jacobian(z, L, Gamma, Gamma_pinv, n, N)
        dz = np.linalg.solve(J, -r)

        # optional damping
        z = z + dz
        print(it, norm_r, np.linalg.norm(dz))
    raise RuntimeError("Newton did not converge")

def powell_hybrid_solve(z0, L, b_ext, Gamma, Gamma_pinv, n, N,
                        tol=1e-10, maxit=50, use_jac=True, verbose = True):
    """
    Powell hybrid method (MINPACK hybr) via scipy.optimize.root.
    """

    def fun(z):
        return residual(z, L, b_ext, Gamma, Gamma_pinv, n, N)

    jac = (lambda z: jacobian(z, L, Gamma, Gamma_pinv, n, N)) if use_jac else None

    sol = root(
        fun, z0,
        jac=jac,
        method="hybr",
        tol=tol,
        options={"maxfev": maxit * (len(z0) + 1)}
    )

    if not sol.success:
        raise RuntimeError(f"Powell hybrid did not converge: {sol.message}")

    if verbose : print("Converged", sol.nfev, np.linalg.norm(sol.fun))
    return sol.x


In [ ]:
n   = 2 # number of modes
N_H = 8 # number of harmonics
# Number of instant for AFT
N = 900*N_H
# Uniform grid
omega = 4.0
nu = 4
tau_j = np.linspace(0, 100*np.pi, N)

z_0 = np.pad(z_0_init, (0,n*(2*N_H+1)-len(z_0_init)), constant_values=0.0)

M = np.array([[1/4, 0.0],
              [0.0, 1/16]])
C = np.array([[0.0, 0.0],
              [0.0, 0.5*kappa_b_p]]) 
K = np.eye(2)
L = build_L(M, C, K, N_H, omega, nu)
Gamma = build_Gamma(tau_j, N_H, n, omega, nu)
Gamma_inverse = np.linalg.pinv(Gamma)
b_ext = build_b_ext(N_H, eps_d_p, nu)

In [ ]:
#z_sol = newton_solve(z_0, L, b_ext, Gamma, Gamma_inverse, n, N, maxit=50)
z_sol = powell_hybrid_solve(z_0, L, b_ext, Gamma, Gamma_inverse, n, N)

In [ ]:
def plot_phase_space(z_solution, N_H):
    """
    z_solution: array of length 2*(2*N_H+1)
    """

    # time grid (one period)
    t = np.linspace(0, 100, 1000)

    # split DOFs
    za = z_solution[::2]        # memory
    zb = z_solution[1::2]        # buffer

    # initialize
    x_a = np.zeros_like(t)
    p_a = np.zeros_like(t)
    x_b = np.zeros_like(t)
    p_b = np.zeros_like(t)

    # DC terms
    x_a += za[0] / np.sqrt(2)
    x_b += zb[0] / np.sqrt(2)

    # harmonics
    for k in range(1, N_H + 1):
        s_idx = 1 + 2*(k-1)
        c_idx = 2 + 2*(k-1)

        # memory (a)
        x_a += za[s_idx]*np.sin(k*t) + za[c_idx]*np.cos(k*t)
        p_a += k/2*za[s_idx]*np.cos(k*t) - k/2*za[c_idx]*np.sin(k*t)

        # buffer (b)
        x_b += zb[s_idx]*np.sin(k*t) + zb[c_idx]*np.cos(k*t)
        p_b += k/4*zb[s_idx]*np.cos(k*t) - k/4*zb[c_idx]*np.sin(k*t)

    a = (x_a + 1j*p_a)/np.sqrt(2)
    a_R = a*np.exp(2j*t)

    b = (x_b + 1j*p_b)/np.sqrt(2)
    b_R = b*np.exp(4j*t)

    # ---- plots ----
    plt.figure()
    plt.plot(np.real(a_R), np.imag(a_R))
    plt.xlabel(r"$\Re(a_R)$")
    plt.ylabel(r"$\Im(a_R)$")
    plt.title("Phase portrait: memory (a)")
    plt.grid(True)
    plt.gca().set_aspect('equal')

    plt.figure()
    plt.plot(np.real(b_R), np.imag(b_R))
    plt.xlabel(r"$\Re(b_R)$")
    plt.ylabel(r"$\Im(b_R)$")
    plt.title("Phase portrait: buffer (b)")
    plt.grid(True)
    plt.gca().set_aspect('equal')
    plt.show()

In [ ]:
z_sol

In [ ]:
plot_phase_space(z_sol, N_H)

In [ ]:
Re_a_list = [-0.2777777777777778]#np.linspace(-0.5,0.5,10)#
Im_a_list = [0.16666666666666663]#np.linspace(-0.5,0.5,10)#
n   = 2 # number of modes
N_H = 8 # number of harmonics
# Number of instant for AFT
N = 900*N_H
# Uniform grid
omega = 4.0
nu = 4
tau_j = np.linspace(0, 100*np.pi, N)

M = np.array([[1.0/4, 0.0],
              [0.0, 1/16]])
C = np.array([[0.0, 0.0],
              [0.0, 0.5*kappa_b_p]]) 
K = np.eye(n)
L = build_L(M, C, K, N_H, omega, nu)
Gamma = build_Gamma(tau_j, N_H, n, omega, nu)
Gamma_inverse = np.linalg.pinv(Gamma)
b_ext = build_b_ext(N_H, eps_d_p, nu)
z_sol_list=[]
for re_a in Re_a_list:
    print(re_a)
    for im_a in Im_a_list:
        z_0_init = np.array([ 
            1.67375389e-01,  2.87830339e-01,
            re_a, im_a, re_a, im_a,
            6.92620826e-01, 0.0, im_a,  0.0,
            re_a, im_a, re_a, im_a,
            0.0, 0.0, 0.0,  5.69644528e-01,
            re_a, im_a, re_a, im_a,
            -9.22066305e-03, im_a, -1.18157915e-03, re_a,
            ])
        z_0 = np.pad(z_0_init, (0,n*(2*N_H+1)-len(z_0_init)), constant_values=0.01)
        
        try:
            #z_sol = newton_solve(
            #    z_0, L, b_ext, Gamma, Gamma_inverse,
            #    n, N, maxit=50
            #)
            z_sol = powell_hybrid_solve(z_0, L, b_ext, Gamma, Gamma_inverse, n, N, verbose = False)
            za = z_sol[::2]   # mode a
            za_1 = [round(coef, 2) for coef in za[3:5]]
            if za_1 == [0.48,  0.02]: 
                print(re_a, im_a)
            z_sol_list.append(z_sol)
        except Exception as e:
            print(e)
            pass

In [ ]:
z_sol_list

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

ax_a, ax_b = axes

for z_sol in z_sol_list:

    if z_sol is None:
        continue

    # one HB period
    t = np.linspace(0, 2*np.pi, 400)

    # split DOFs (time-major Fourier ordering)
    za = z_sol[::2]   # mode a
    zb = z_sol[1::2]  # mode b

    # initialize
    x_a = np.zeros_like(t)
    p_a = np.zeros_like(t)
    x_b = np.zeros_like(t)
    p_b = np.zeros_like(t)

    # DC
    x_a += za[0] / np.sqrt(2)
    x_b += zb[0] / np.sqrt(2)

    # harmonics
    for k in range(1, N_H + 1):
        s_idx = 1 + 2*(k-1)
        c_idx = 2 + 2*(k-1)

        # mode a
        x_a += za[s_idx]*np.sin(k*t) + za[c_idx]*np.cos(k*t)
        p_a += k/2*za[s_idx]*np.cos(k*t) - k/2*za[c_idx]*np.sin(k*t)

        # mode b  (remember physical frequency factor 1/2 in your model)
        x_b += zb[s_idx]*np.sin(k*t) + zb[c_idx]*np.cos(k*t)
        p_b += (k/4)*zb[s_idx]*np.cos(k*t) - (k/4)*zb[c_idx]*np.sin(k*t)

    # complex amplitudes
    a = (x_a + 1j*p_a)/np.sqrt(2)
    b = (x_b + 1j*p_b)/np.sqrt(2)

    a = (x_a + 1j*p_a)/np.sqrt(2)
    a_R = a*np.exp(2j*t)

    b = (x_b + 1j*p_b)/np.sqrt(2)
    b_R = b*np.exp(4j*t)

    ax_a.plot(np.real(a_R), np.imag(a_R))
    ax_b.plot(np.real(b_R), np.imag(b_R))

# formatting
ax_a.set_title("Phase portrait: mode a")
ax_a.set_xlabel(r"$\Re(a_R)$")
ax_a.set_ylabel(r"$\Im(a_R)$")
ax_a.set_xlim([-1,1])
ax_a.set_ylim([-1,1])
ax_a.set_aspect('equal')
ax_a.grid(True)

ax_b.set_title("Phase portrait: mode b")
ax_b.set_xlabel(r"$\Re(b_R)$")
ax_b.set_ylabel(r"$\Im(b_R)$")
#ax_b.set_xlim([-.2,.5])
#ax_b.set_ylim([-.2,.2])
ax_b.set_aspect('equal')
ax_b.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
za_1_list=[]
for z_sol in z_sol_list:
    if z_sol is None:
        continue
    za = z_sol[::2]   # mode a
    za_1 = [round(coef, 2) for coef in za[3:5]]
    za_1_list.append(za_1)

In [ ]:
1/np.sqrt(2)

In [ ]:
np.unique(np.array(za_1_list), axis = 0)

# Checks

In [ ]:
import numpy as np

def build_Phi_terms(t, N_H):
    """Return Phi, dPhi, ddPhi for basis [1/sqrt2, sin, cos, ...]"""
    t = np.asarray(t)
    N = t.size
    Phi = np.zeros((N, 2*N_H+1))
    dPhi = np.zeros_like(Phi)
    ddPhi = np.zeros_like(Phi)

    Phi[:, 0] = 1/np.sqrt(2)

    for k in range(1, N_H+1):
        is_ = 1 + 2*(k-1)
        ic_ = 2 + 2*(k-1)

        Phi[:, is_] = np.sin(k*t)
        Phi[:, ic_] = np.cos(k*t)

        dPhi[:, is_] =  k*np.cos(k*t)
        dPhi[:, ic_] = -k*np.sin(k*t)

        ddPhi[:, is_] = -(k**2)*np.sin(k*t)
        ddPhi[:, ic_] = -(k**2)*np.cos(k*t)

    return Phi, dPhi, ddPhi

def build_Gamma_from_Phi(Phi, n):
    return np.kron(Phi, np.eye(n))

def check_L_vs_Gamma(L, t, N_H, n, M, C, K, trials=5, tol=1e-10):
    N = len(t)
    Phi, dPhi, ddPhi = build_Phi_terms(t, N_H)

    Gamma   = build_Gamma_from_Phi(Phi, n)
    Gamma_d = build_Gamma_from_Phi(dPhi, n)
    Gamma_dd= build_Gamma_from_Phi(ddPhi, n)

    # time-domain linear operator on stacked vectors
    Mbig = np.kron(np.eye(N), M)  # (nN x nN) but note stacking convention!
    Cbig = np.kron(np.eye(N), C)
    Kbig = np.kron(np.eye(N), K)

    # IMPORTANT: above assumes time-major stacking [x(t1); x(t2); ...]
    # If your tilde x is DOF-major [x1(t1..tN); x2(t1..tN)], use this instead:
    #Mbig = np.kron(M, np.eye(N))
    #Cbig = np.kron(C, np.eye(N))
    #Kbig = np.kron(K, np.eye(N))

    for r in range(trials):
        z = np.random.randn(n*(2*N_H+1))

        x_tilde   = Gamma   @ z
        dx_tilde  = Gamma_d @ z
        ddx_tilde = Gamma_dd@ z

        # time-domain linear result
        lin_time = Mbig @ ddx_tilde + Cbig @ dx_tilde + Kbig @ x_tilde

        # Fourier-domain linear result mapped back to time
        lin_freq_time = Gamma @ (L @ z)

        err = np.linalg.norm(lin_time - lin_freq_time) / max(1.0, np.linalg.norm(lin_time))
        print(f"trial {r}: relative error = {err:.3e}")

        if err > tol:
            print("❌ Inconsistent (likely ordering or sign/scaling mismatch).")
            return False

    print("✅ L is consistent with Gamma (within tolerance).")
    return True


In [ ]:
n = 2
N_H = 5
N = 16*N_H
t = np.linspace(0, 2*np.pi, N, endpoint=False)  # or your period

# put your matrices here
M = np.array([[1.0, 0.0],[0.0, 0.25]])
C = np.array([[0.0, 0.0],[0.0, 0.5*kappa_b_p]])
K = np.eye(2)

L = build_L(M, C, K, N_H)  # make sure your build_L uses (M,C,K) in this order

check_L_vs_Gamma(L, t, N_H, n, M, C, K)


# Floquet coefficients

In [ ]:
def Floq_coef(z_sol, L, M, C, N_H, n):

    Delta_1 = np.zeros([(2*N_H + 1)*n, (2*N_H + 1)*n])

    # Put C on every diagonal block (including k=0)
    for i in range(2*N_H + 1):
        Delta_1[i*n:(i+1)*n, i*n:(i+1)*n] = C

    # Put ±2k M between the (cos,sin) blocks for each k=1..N_H
    for k in range(1, N_H + 1):
        i = 2*k - 1  # cos index
        j = 2*k      # sin index

        # upper diagonal (cos -> sin)
        Delta_1[i*n:(i+1)*n, j*n:(j+1)*n] = -2*k * M

        # lower diagonal (sin -> cos)
        Delta_1[j*n:(j+1)*n, i*n:(i+1)*n] =  2*k * M


    Delta_2 = block_diag(*([M] * (2*N_H + 1)))
    h_z = jacobian(z_sol, L, Gamma, Gamma_inverse, n, N)

    B_1 = np.block([[Delta_1, h_z],[-np.eye((2*N_H + 1)*n),np.zeros([(2*N_H + 1)*n, (2*N_H + 1)*n])]])
    B_2 = -np.block([[Delta_2, np.zeros([(2*N_H + 1)*n, (2*N_H + 1)*n])],[np.zeros([(2*N_H + 1)*n, (2*N_H + 1)*n]), np.eye((2*N_H + 1)*n)]])
    B = np.linalg.inv(B_2)@B_1
    eigvals = np.linalg.eigvals(B)
    # sort by absolute value of imaginary part
    idx = np.argsort(np.abs(np.imag(eigvals)))
    # keep 2n smallest
    eigvals_selected = eigvals[idx[:2*n]]
    return(eigvals_selected)

In [ ]:
z_guess = np.array([ 1.09656245e-01,  1.88572491e-01,  6.36702210e-03,  8.76868903e-03,
         6.72303249e-03,  9.24024143e-03,  2.40775936e-01, -5.19021061e-17,
         1.29455705e-02,  2.55713260e-16, -1.36877767e-03,  6.68868671e-03,
         1.13576613e-03, -5.62391811e-03,  2.66417683e-04, -2.31832679e-02,
        -4.52413407e-03,  3.62199837e-01, -1.04515526e-04, -1.66109841e-03,
        -1.20309391e-04, -1.94500316e-03, -5.97721955e-03, -6.57545126e-02,
        -5.72870310e-04, -6.60436919e-03, -3.64503331e-06, -3.42114325e-05,
         7.05922864e-07,  6.51141406e-06,  2.84776137e-04,  2.49632530e-03,
        -2.19458382e-03, -1.88634782e-02])
z_guess = np.array([ 1.18006838e-01,  2.02932751e-01, -3.34023533e-02, -4.59062405e-02,
         3.34138564e-02,  4.60150836e-02,  4.76099915e-01,  4.23704307e-17,
         2.32459958e-02,  2.01909741e-16, -2.19830761e-02,  1.08803672e-01,
        -2.49643296e-02,  1.21950474e-01,  2.96288467e-04, -3.98836442e-02,
        -3.52762253e-03,  4.02809728e-01, -4.80658168e-04, -7.76802484e-03,
         3.98193189e-04,  6.32559370e-03, -6.49672672e-03, -7.14528852e-02,
        -9.54875632e-04, -1.08346241e-02, -5.86588812e-04, -5.47809788e-03,
        -7.96556265e-04, -7.48934796e-03,  4.38435272e-04,  3.81730578e-03,
        -2.18421057e-03, -1.87709435e-02])
z_guess = np.array([8.66873728e-02,  1.49073624e-01, -3.40735334e-02, -4.68288048e-02,
  3.39892906e-02,  4.68076623e-02,  3.32323386e-01,  7.16053497e-17,
  9.64053733e-03,  1.52042243e-16, -2.30275855e-02, 1.13930326e-01,
 -2.48028632e-02,  1.21123784e-01,  2.24039230e-04, -1.80297585e-02,
 -4.73505058e-03,  3.04585535e-01, -3.56052588e-04,-5.75975495e-03,
  3.35670847e-04,  5.33902982e-03, -4.76897548e-03, -5.24651129e-02,
 -4.10866267e-04, -4.76085580e-03, -6.77612211e-04, -6.33175583e-03,
 -8.00994709e-04, -7.53372922e-03,  1.92956610e-04, 1.69434713e-03,
 -1.62052258e-03, -1.39295292e-02])

eps_p_list = np.concatenate((
    np.linspace(1.2, 0.7, 10, endpoint=False),
    np.linspace(0.7, 0.58, 100)
))
eigvals_list = []
n   = 2 # number of modes
N_H = 12 # number of harmonics
# Number of instant for AFT
N = 900*N_H
# Uniform grid
omega = 4.0
nu = 4
tau_j = np.linspace(0, 100*np.pi, N)

z_guess = np.pad(z_guess, (0,n*(2*N_H+1)-len(z_guess)), constant_values=0.0)
for epsilon_p in eps_p_list[::]:
    print(epsilon_p)
    g2 = jv(1,epsilon_p)*E_J*phi_a**2*phi_b # (Leon's paper)
    epsilon_d = 10.12 *g2#0.75*g2_max

    varphi_a, varphi_b = np.sqrt(Z_a)/np.sqrt(2)/phi_quantum, np.sqrt(Z_b)/np.sqrt(2)/phi_quantum
    beta_a_p, beta_b_p = 2*E_J*np.sin(epsilon_p)*varphi_a**2/(w_a), 2*E_J*np.sin(epsilon_p)*varphi_b**2/(w_b)
    B = np.array([beta_a_p, beta_b_p])  # shape (2,)
    eps_d_p = epsilon_d*varphi_b/w_b
    kappa_b_p = kappa_b / w_b

    M = np.array([[1/4, 0.0],
                [0.0, 1/16]])
    C = np.array([[0.0, 0.0],
                [0.0, 0.5*kappa_b_p]]) 
    K = np.eye(2)
    L = build_L(M, C, K, N_H, omega, nu)
    Gamma = build_Gamma(tau_j, N_H, n, omega, nu)
    Gamma_inverse = np.linalg.pinv(Gamma)
    b_ext = build_b_ext(N_H, eps_d_p, nu)
    try :
        z_sol = powell_hybrid_solve(z_guess, L, b_ext, Gamma, Gamma_inverse, n, N, verbose = False)
        eigvals_list.append(Floq_coef(z_sol, L, M, C, N_H, n))
        plot_phase_space(z_sol, N_H)
        z_guess = z_sol
    except Exception as e:
        pass
eigvals_array = np.array(eigvals_list)


In [ ]:
T=2*np.pi/(omega/4) # Period corresponding to the memory frequency (one fourth of the drive period)
Poincare_eig = np.exp(eigvals_array*T)

In [ ]:
Poincare_eig.T.shape

In [ ]:
plt.figure()
for i in range(2*n):
    plt.plot(eps_p_list[:], np.real(eigvals_array[:, i]))
plt.xlabel("Pump amplitude "+r"$\varepsilon_p$")
plt.ylabel("Real part of Floquet\n exponents")
plt.show()

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
import Plot_setting
from Plot_setting import *
plt.figure()
for i in range(2*n):
    plt.plot(eps_p_list[:], np.imag(eigvals_array[:, i]))
plt.xlabel("Pump amplitude "+r"$\varepsilon_p$")
plt.ylabel("Imaginary part of Floquet\n exponents")
plt.show()

In [ ]:
plt.figure()
for i in range(2*n):
    plt.plot(eps_p_list[:], np.real(Poincare_eig[:, 3-i]))
plt.axhline([1], ls="--", color="r")
plt.xlabel("Pump amplitude "+r"$\varepsilon_p$")
plt.ylabel("Real part of Poincaré \n eigenvalues")
plt.show()

In [ ]:
plt.figure()
for i in range(2*n):
    plt.plot(eps_p_list[:], np.abs(Poincare_eig[:, i]))
#plt.axhline([-1], ls="--", color="r")
plt.xlabel("Pump amplitude "+r"$\varepsilon_p$")
plt.ylabel("Absolute value of Poincaré \n eigenvalues")
plt.show()

# Loop over the drive amplitude

In [ ]:
z_guess_0 = np.array([ 1.34099052e-01,  2.30606040e-01,  1.25652573e-02,  1.72695817e-02,
        -1.21191575e-02, -1.66902312e-02,  5.61246549e-01, -1.63084347e-16,
         8.36765850e-02,  2.49942258e-16,  6.77751048e-03, -3.36271519e-02,
         1.02730741e-02, -5.02563652e-02,  3.18872831e-04, -1.32079967e-01,
        -1.15687881e-03,  4.33513593e-01,  2.44655499e-04,  3.94124588e-03,
        -1.08939725e-04, -1.71527411e-03, -6.84650527e-03, -7.51874801e-02,
        -3.24849684e-03, -3.60949599e-02,  1.03399291e-04,  9.59466433e-04,
         3.44702208e-04,  3.23640116e-03,  1.42436411e-03,  1.22930940e-02,
        -2.11452028e-03, -1.81502800e-02])

eps_d_ratio_list = np.linspace(4, 13.5, 20)
eps_p_list = np.linspace(0.55, 0.7, 20)

Nd = len(eps_d_ratio_list)
Np = len(eps_p_list)

# preallocate eigenvalue storage
eigvals_array = np.full((Nd, Np, 2*n), np.nan, dtype=complex)


for j, epsilon_p in enumerate(eps_p_list):
    print(j)

    z_guess = z_guess_0

    g2 = jv(1, epsilon_p)*E_J*phi_a**2*phi_b
    varphi_a = np.sqrt(Z_a)/np.sqrt(2)/phi_quantum
    varphi_b = np.sqrt(Z_b)/np.sqrt(2)/phi_quantum
    beta_a_p = 2*E_J*np.sin(epsilon_p)*varphi_a**2/(w_a)
    beta_b_p = 2*E_J*np.sin(epsilon_p)*varphi_b**2/(w_b)
    B = np.array([beta_a_p, beta_b_p])  # shape (2,)
    eps_d_p_base = varphi_b/w_b
    kappa_b_p = kappa_b / w_b

    M = np.array([[1/4, 0.0],
                  [0.0, 1/16]])
    C = np.array([[0.0, 0.0],
                  [0.0, 0.5*kappa_b_p]])
    K = np.eye(2)

    L_base = build_L(M, C, K, N_H, omega, nu)
    Gamma = build_Gamma(tau_j, N_H, n, omega, nu)
    Gamma_inverse = np.linalg.pinv(Gamma)

    for i, eps_d_ratio in enumerate(eps_d_ratio_list):

        epsilon_d = eps_d_ratio * g2
        eps_d_p = epsilon_d * eps_d_p_base

        b_ext = build_b_ext(N_H, eps_d_p, nu)

        try:
            z_sol = powell_hybrid_solve(
                z_guess, L_base, b_ext,
                Gamma, Gamma_inverse,
                n, N, verbose=False
            )

            eigvals_array[i, j, :] = Floq_coef(
                z_sol, L_base, M, C, N_H, n
            )
            z_guess = z_sol

        except Exception:
            pass

In [ ]:
T=2*np.pi/(omega/4) # Period corresponding to the memory frequency (one fourth of the drive period)
Poincare_eig = np.exp(eigvals_array*T)
score = np.max(np.abs(Poincare_eig), axis=2)      # (Nd, Np)

j_max = np.nanargmax(score, axis=1)
epsilon_p_max_array = eps_p_list[j_max]

epsilon_p_max_array

In [ ]:
T=2*np.pi/(omega/4) # Period corresponding to the memory frequency (one fourth of the drive period)
Poincare_eig = np.exp(eigvals_array*T)
tol=1e-2
mask = np.any(
    np.abs(Poincare_eig) > 1,
    axis=2
)   # shape (Nd, Np)

epsilon_p_doubling_list = []

for i in range(Nd):
    idx = np.where(mask[i])[0]
    if len(idx) > 0:
        epsilon_p_doubling_list.append(eps_p_list[idx[0]])
    else:
        epsilon_p_doubling_list.append(np.nan)

epsilon_p_doubling_array = np.array(epsilon_p_doubling_list)
epsilon_p_doubling_array

In [ ]:
plt.figure()
plt.plot(eps_d_ratio_list, jv(1,np.array(epsilon_p_doubling_array))*E_J*phi_a**2*phi_b/2/np.pi*1e3)
plt.xlabel(r"$\varepsilon_d/g_2$")
plt.ylabel("Value of "+r"\varepsilon_p"+" at which period doubling occurs")
plt.show()

In [ ]:
import json 
with open("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/analysis/output/epsilon_zero_list_12_02_2026_Hill_unstable.json", "w") as f:
    json.dump(list(epsilon_p_doubling_array), f)